# Endymion Viva Demo
## Lunar Terrain Hazard Assessment and Hazard-Aware Navigation

This notebook demonstrates the final Endymion pipeline by:
- loading saved artefacts from the official CLI runners,
- visualising terrain, hazard, and navigation outputs,
- summarising benchmark comparisons,
- inspecting the machine-learning crater-probability output.

This notebook is intended for the final video demonstration and viva.

## Demo flow

This notebook follows the same structure as the final Endymion system:

1. Load saved ROI artefacts
2. Inspect terrain-derived products
3. Visualise hazard and planned route
4. Review benchmark comparison outputs
5. Review ML bridge outputs
6. Summarise what the system demonstrates

The CLI runners remain the primary execution interface:
- `python -m src.run_endymion`
- `python benchmark\run_benchmark.py`
- `python -m src.run_crater_ml`

### Setup and Paths

In [ ]:
# References / notes:
# - Paths and artefact names are based on the final Endymion pipeline contract.
# - Main project runners:
#     * python -m src.run_endymion
#     * python benchmark\run_benchmark.py
#     * python -m src.run_crater_ml
# - Saved artefacts follow the report's final pipeline structure:
#     derived terrain artefacts, crater artefacts, navigation artefacts,
#     evaluation artefacts, ML artefacts, and benchmark summaries.

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Main base paths
PERSISTENT_DIR = Path(r"C:\Endymion\persistent")
RUNTIME_DIR = Path(r"C:\Endymion\runtime")

# Canonical ROI path used in the final report
ROI_DIR = PERSISTENT_DIR / "derived" / "ldem_80s_20m" / "roi_14688_15712_14688_15712"

# Example run directories
RUN_DIR = ROI_DIR / "navigation" / "terrain_only_smoke_test_v2"
BENCHMARK_DIR = ROI_DIR / "benchmarks" / "appendix_benchmark"
ML_SUMMARY_PATH = ROI_DIR / "crater_proba_ml_v1_summary.json"

print("ROI_DIR:", ROI_DIR)
print("RUN_DIR:", RUN_DIR)
print("BENCHMARK_DIR:", BENCHMARK_DIR)
print("ML_SUMMARY_PATH:", ML_SUMMARY_PATH)

### Check Artifact Existance 

In [ ]:
# Quick sanity check of important files before visualisation

expected_files = {
    "DEM": ROI_DIR / "dem_m.npy",
    "Slope": ROI_DIR / "slope_deg.npy",
    "Roughness": ROI_DIR / "roughness_rms.npy",
    "Crater mask": ROI_DIR / "crater_mask.npy",
    "ML crater probability": ROI_DIR / "crater_proba_ml_v1.npy",
    "Benchmark summary CSV": BENCHMARK_DIR / "summary.csv",
}

for label, path in expected_files.items():
    print(f"{label:24} -> {'FOUND' if path.exists() else 'MISSING'} | {path}")

### Load Core Terrain Artefacts 

In [ ]:
# Load terrain-derived artefacts

dem_m = np.load(ROI_DIR / "dem_m.npy")
slope_deg = np.load(ROI_DIR / "slope_deg.npy")
roughness_rms = np.load(ROI_DIR / "roughness_rms.npy")

print("DEM shape:", dem_m.shape, dem_m.dtype)
print("Slope shape:", slope_deg.shape, slope_deg.dtype)
print("Roughness shape:", roughness_rms.shape, roughness_rms.dtype)

### Visualise terrain products

In [ ]:
# Visual 1: DEM
plt.figure(figsize=(7, 6))
plt.imshow(dem_m)
plt.title("Canonical ROI DEM (metres)")
plt.colorbar(label="Elevation (m)")
plt.tight_layout()
plt.show()

In [ ]:
# Visual 2: Slope
plt.figure(figsize=(7, 6))
plt.imshow(slope_deg)
plt.title("Slope (degrees)")
plt.colorbar(label="Slope (deg)")
plt.tight_layout()
plt.show()

In [ ]:
# Visual 3: Roughness
plt.figure(figsize=(7, 6))
plt.imshow(roughness_rms)
plt.title("RMS Roughness")
plt.colorbar(label="Roughness")
plt.tight_layout()
plt.show()

### Load and Visualise Navigation Rrun

In [ ]:
# Load one completed navigation run
hazard = np.load(RUN_DIR / "hazard.npy") # Hazard map used for path planning
cost = np.load(RUN_DIR / "cost.npy") # Cost map used for path planning 
path_rc = np.load(RUN_DIR / "path_rc.npy") # Path nodes in row/column coordinates

with open(RUN_DIR / "nav_meta.json", "r", encoding="utf-8") as f:
    nav_meta = json.load(f)

metrics_path = RUN_DIR / "metrics.json"
metrics = None
if metrics_path.exists():
    with open(metrics_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)

print("Hazard shape:", hazard.shape)
print("Cost shape:", cost.shape)
print("Path nodes:", len(path_rc))
print("Success:", nav_meta.get("result", {}).get("success", None))

Plot Path Overlay 

In [ ]:
# Visual: hazard map with planned path

plt.figure(figsize=(7, 6))
plt.imshow(hazard)
plt.title("Hazard Map with Planned Route")
plt.colorbar(label="Hazard")

if len(path_rc) > 0:
    plt.plot(path_rc[:, 1], path_rc[:, 0], linewidth=2)

plt.tight_layout()
plt.show()

### Show Compact Metrics Summary 

In [ ]:
# Compact summary of one run

if metrics is not None:
    status = metrics.get("status", {})
    geometry = metrics.get("geometry", {})
    safety = metrics.get("safety", {})
    efficiency = metrics.get("efficiency", {})

    print("=== Run Summary ===")
    print("Success:", status.get("success"))
    print("Failure reason:", status.get("failure_reason"))
    print("Path length (m):", geometry.get("path_length_m"))
    print("Mean path hazard:", safety.get("path_hazard_mean"))
    print("Max path hazard:", safety.get("path_hazard_max"))
    print("Cost per metre:", efficiency.get("cost_per_m"))
    print("Expansions:", efficiency.get("expansions"))
else:
    print("metrics.json not found in RUN_DIR.")

### Benchmark Summary Section

In [ ]:
# Load benchmark summary
benchmark_csv = BENCHMARK_DIR / "summary.csv"

if benchmark_csv.exists():
    bench_df = pd.read_csv(benchmark_csv)
    bench_df
else:
    print("Benchmark summary.csv not found.")

In [ ]:
# Show only the most relevant benchmark columns
important_cols = [
    "hazard_model_id",
    "success_rate",
    "mean_path_hazard_mean",
    "mean_cost_per_m",
    "mean_expansions"
]

existing_cols = [c for c in important_cols if c in bench_df.columns]
bench_df[existing_cols]